# Basic quantum logic gates

A one-qubit gate is a $2\times 2$ unitary. A two-qubit gate is a $4\times 4$
unitary. This notebook walks through the gates you will see in every later
snippet: **X, Y, Z, H, S, T, CX, SWAP**.

Qiskit draws bitstrings with **qubit 0 on the right**.

This notebook is self-contained. It does not import `logic_gates.py`.

In [5]:
import qiskit as qk
import qiskit_aer as qka


In [6]:
def show(qc, title):
    print(title)
    print(qc.draw())
    sv = qk.quantum_info.Statevector.from_instruction(
        qc.remove_final_measurements(inplace=False)
    )
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()

## Pauli X is a quantum NOT

$X|0\rangle = |1\rangle$ and $X|1\rangle = |0\rangle$. The matrix is
$\sigma_x = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$.

In [7]:
x_circ = qk.QuantumCircuit(1)
x_circ.x(0)
show(x_circ, "X on |0>")
print("matrix:\n", qk.quantum_info.Operator(x_circ).data.round(3))

X on |0>
   ┌───┐
q: ┤ X ├
   └───┘
  +1.000+0.000j |1>

matrix:
 [[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]


## Z and Y, and the phase gates S and T

- $Z$ flips the sign of $|1\rangle$ and leaves $|0\rangle$ alone.
- $S = \sqrt{Z}$ and $T = \sqrt{S}$ are finer phase ticks.
- $Y = iXZ$ rotates around the $y$ axis.

In [8]:
plus = qk.QuantumCircuit(1)
plus.h(0)
show(plus, "H|0> = |+>")

z_plus = qk.QuantumCircuit(1)
z_plus.h(0)
z_plus.z(0)
show(z_plus, "Z|+> = |->")

s_plus = qk.QuantumCircuit(1)
s_plus.h(0)
s_plus.s(0)
show(s_plus, "S|+>")

y_zero = qk.QuantumCircuit(1)
y_zero.y(0)
show(y_zero, "Y|0>")

H|0> = |+>
   ┌───┐
q: ┤ H ├
   └───┘
  +0.707+0.000j |0>
  +0.707+0.000j |1>

Z|+> = |->
   ┌───┐┌───┐
q: ┤ H ├┤ Z ├
   └───┘└───┘
  +0.707+0.000j |0>
  -0.707+0.000j |1>

S|+>
   ┌───┐┌───┐
q: ┤ H ├┤ S ├
   └───┘└───┘
  +0.707+0.000j |0>
  +0.000+0.707j |1>

Y|0>
   ┌───┐
q: ┤ Y ├
   └───┘
  +0.000+1.000j |1>



## Hadamard creates equal superposition

$H|0\rangle = (|0\rangle+|1\rangle)/\sqrt{2}$. Measuring many times
gives a 50/50 histogram.

In [9]:
h = qk.QuantumCircuit(1, 1)
h.h(0)
h.measure(0, 0)
sim = qka.AerSimulator()
print(sim.run(qk.transpile(h, sim), shots=2000).result().get_counts())

{'1': 1014, '0': 986}


## CX and SWAP

CX flips the target when the control is $|1\rangle$. SWAP exchanges two
qubits. Both are written out as a full computational-basis table so you
can see every input.

In [10]:
print("CX  control=q1  target=q0")
for c, t in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    qc = qk.QuantumCircuit(2)
    if t:
        qc.x(0)
    if c:
        qc.x(1)
    qc.cx(1, 0)
    out = next(iter(qk.quantum_info.Statevector.from_instruction(qc).to_dict()))
    print(f"  |{c}{t}> -> |{out}>")

print("\nSWAP |10> -> |01>")
sw = qk.QuantumCircuit(2)
sw.x(1)
sw.swap(0, 1)
show(sw, "after SWAP")

CX  control=q1  target=q0
  |00> -> |00>
  |01> -> |01>
  |10> -> |11>
  |11> -> |10>

SWAP |10> -> |01>
after SWAP
             
q_0: ──────X─
     ┌───┐ │ 
q_1: ┤ X ├─X─
     └───┘   
  +1.000+0.000j |01>

